# Try SparseCore in Colab

This notebook is the fastest way to try SparseCore without installing anything on your own machine. It installs the library from PyPI, verifies the install works, and runs a small sparse training loop end-to-end.

Colab runtime you need: **CPU** (the default). No GPU required — SparseCore is CPU-only for v0.1.

Links:
- Repository: https://github.com/DarshanFofadiya/sparsecore
- Docs: https://github.com/DarshanFofadiya/sparsecore/tree/main/docs
- Issues: https://github.com/DarshanFofadiya/sparsecore/issues

## 1. Install SparseCore

This pulls the pre-built Linux x86_64 wheel from PyPI. Should take under 30 seconds — no compiler needed, libomp is bundled inside the wheel.

In [ ]:
!pip install sparsecore

## 2. Verify the install

Import the library and run a quick forward + backward pass. If this cell prints `torch.Size([256, 32])` and finishes without error, the install is good.

In [ ]:
import torch
import sparsecore

print(f"sparsecore {sparsecore.__version__}")
print(f"torch      {torch.__version__}")

# Build a random 90% sparse (256 x 128) weight matrix.
W = sparsecore.PaddedCSR.random(256, 128, sparsity=0.9, seed=0)

# Dense input (128, 32) with autograd tracking.
X = torch.randn(128, 32, requires_grad=True)

# Forward pass via our sparse kernel.
Y = sparsecore.spmm(W, X)
print(f"output shape: {Y.shape}")

# Backward pass. X.grad should flow through our autograd-aware SpMM.
Y.sum().backward()
print(f"X.grad shape: {X.grad.shape}, finite: {torch.isfinite(X.grad).all().item()}")

## 3. Train a tiny MLP with SET topology mutation

This is the DST-researcher use case: drop-in `SparseLinear`, attach a `SparsityAlgorithm` (SET in this case), training loop is otherwise standard PyTorch.

We fit a tiny 2-layer MLP on random regression targets just to show the machinery works. Real datasets live in `examples/demo_05_mnist.py` and `examples/demo_15_mini_gpt.py` in the repo.

In [ ]:
import torch
import sparsecore

torch.manual_seed(0)

# Two-layer MLP. The first layer is 90% sparse; the second is standard dense.
model = torch.nn.Sequential(
    sparsecore.SparseLinear(64, 128, sparsity=0.9),
    torch.nn.ReLU(),
    torch.nn.Linear(128, 8),
)
opt = torch.optim.SGD(model.parameters(), lr=0.05)

# SET: drop the smallest 30% of weights every 50 steps and regrow at random.
algo = sparsecore.SET(sparsity=0.9, drop_fraction=0.3, update_freq=50)
model.apply(algo)

# Random regression target just to give the model something to fit.
X_train = torch.randn(512, 64)
y_train = torch.randn(512, 8)

losses = []
for step in range(200):
    idx = torch.randint(0, 512, (32,))
    x, y = X_train[idx], y_train[idx]
    pred = model(x)
    loss = (pred - y).pow(2).mean()
    loss.backward()
    opt.step()
    algo.step()
    opt.zero_grad()
    if step % 25 == 0:
        losses.append((step, loss.item()))
        print(f"step {step:3d}  loss={loss.item():.4f}")

# Show the topology mutated.
sparse_layer = model[0]
print(f"\nlive connections in sparse layer: {sparse_layer.nnz} "
      f"({sparse_layer.density:.1%} density)")
print(f"SET fired {algo._step_idx // algo.update_freq} times during training")

## 4. Plot the loss curve

Quick sanity check that training actually decreased loss.

In [ ]:
import matplotlib.pyplot as plt

steps, vals = zip(*losses)
plt.figure(figsize=(8, 4))
plt.plot(steps, vals, marker="o")
plt.xlabel("training step")
plt.ylabel("MSE loss")
plt.title("SparseLinear + SET on a toy regression task")
plt.grid(alpha=0.3)
plt.show()

## 5. Write your own SparsityAlgorithm

The whole point of SparseCore is that you can plug in new DST algorithms as short Python subclasses. Here's a minimal example — a "keep-top-K" algorithm that keeps only the largest-magnitude connections per row.

In [ ]:
import numpy as np

class KeepTopK(sparsecore.DynamicSparsityAlgorithm):
    """Every `update_freq` steps, keep only the top-K largest
    magnitude weights per row. Drops everything else permanently
    (no regrowth — a simple demo, not a serious DST algorithm)."""

    def __init__(self, sparsity, k_per_row, update_freq=100):
        # drop_fraction is required by the base class but unused here.
        super().__init__(sparsity=sparsity, drop_fraction=0.0, update_freq=update_freq)
        self.k_per_row = k_per_row

    def update(self):
        for layer in self.layers:
            csr = layer._csr
            col_indices = np.asarray(csr.col_indices)
            values = np.asarray(csr.values)
            row_start = np.asarray(csr.row_start)
            row_nnz = np.asarray(csr.row_nnz)

            for i in range(csr.nrows):
                n_live = int(row_nnz[i])
                if n_live <= self.k_per_row:
                    continue
                start = int(row_start[i])
                vals = values[start:start + n_live]
                cols = col_indices[start:start + n_live]
                top_k_idx = np.argsort(-np.abs(vals))[:self.k_per_row]
                kept_cols = cols[top_k_idx]
                kept_vals = vals[top_k_idx]
                order = np.argsort(kept_cols)
                csr.rewrite_row(i, kept_cols[order].astype(np.int32),
                                kept_vals[order].astype(np.float32))

# Try it.
model2 = torch.nn.Sequential(
    sparsecore.SparseLinear(64, 128, sparsity=0.9),
    torch.nn.ReLU(),
    torch.nn.Linear(128, 8),
)
algo2 = KeepTopK(sparsity=0.9, k_per_row=4, update_freq=20)
model2.apply(algo2)

print(f"before: {model2[0].nnz} live connections")

# Run a few steps so the algorithm fires.
opt2 = torch.optim.SGD(model2.parameters(), lr=0.05)
for _ in range(40):
    x = torch.randn(32, 64)
    model2(x).pow(2).sum().backward()
    opt2.step()
    algo2.step()
    opt2.zero_grad()

print(f"after:  {model2[0].nnz} live connections")
print(f"(expected ≤ {128 * 4} = out_features × k_per_row)")

## What's next

- Read the [honest performance picture](https://github.com/DarshanFofadiya/sparsecore#the-honest-performance-picture) in the README — SparseCore is about memory + pluggability, not raw speed over GPU.
- Browse the [design docs](https://github.com/DarshanFofadiya/sparsecore/tree/main/docs/design) to see how Padded-CSR and the NEON kernels work.
- Look at [`sparsecore/router.py`](https://github.com/DarshanFofadiya/sparsecore/blob/main/sparsecore/router.py) — the built-in `SET` and `RigL` implementations are ~50 lines of real logic each, good templates for writing your own.
- Open an issue if something didn't work as expected. v0.1 is the first public release, so feedback is wanted.